# Fiş testi: PaliGemma 2 3B mix @224 vs @448

Aynı fişlere aynı soruyu (`What is the total amount?`) iki çözünürlükte soruyoruz. Değişen tek şey çözünürlük.

**Başlamadan önce**
1. Colab menüsünden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. Hugging Face'te şu iki sayfaya girip lisansı kabul et (Google'ın modelleri onaylı erişimli):
   - https://huggingface.co/google/paligemma2-3b-mix-224
   - https://huggingface.co/google/paligemma2-3b-mix-448
3. https://huggingface.co/settings/tokens adresinden bir *read* token oluştur, 2. hücrede sorulduğunda yapıştır.

Sonra **Çalışma zamanı → Tümünü çalıştır**. Sonunda `fis_karsilastirma.png` iner ve yazıya yapıştırılacak iki cümle yazdırılır.

In [ ]:
# Colab'daki cudf ile olan pyarrow cakismasini ve gereksiz hata uyarilarini onlemek icin cudf kaldirilir
!pip -q uninstall -y cudf-cu12 2>/dev/null
!pip -q install -U transformers accelerate datasets


In [ ]:
import os
from huggingface_hub import login

# Google Colab Secrets (sol paneldeki anahtar simgesi) icinde HF_TOKEN varsa otomatik oturum acar,
# yoksa interaktif giris widget'i cagirir.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('HF_TOKEN Colab Secrets uzerinden otomatik yuklendi.')
    else:
        from huggingface_hub import notebook_login
        notebook_login()
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()


## Ayarlar
- `KAYNAK = "cord"`: CORD-v2 veri setinden gerçek fişler (doğru toplamlar veri setinde hazır).
- `KAYNAK = "yukle"`: kendi fiş fotoğraflarını yüklersin, doğru toplamları `DOGRU_TOPLAM` sözlüğüne elle yazarsın.

In [ ]:
KAYNAK = "cord"        # "cord" veya "yukle"
FIS_SAYISI = 10        # kaç fiş denensin
SORU = "What is the total amount?"
MODELLER = {224: "google/paligemma2-3b-mix-224", 448: "google/paligemma2-3b-mix-448"}

# KAYNAK = "yukle" ise: dosya adı -> fişte yazan toplam
DOGRU_TOPLAM = {
    # "fis1.jpg": "245,90",
}

In [ ]:
import json, re, io
from PIL import Image

fisler = []   # (isim, PIL.Image, dogru_toplam)

if KAYNAK == "cord":
    from datasets import load_dataset
    ds = load_dataset("naver-clova-ix/cord-v2", split="test")
    for i, ex in enumerate(ds):
        gt = json.loads(ex["ground_truth"])["gt_parse"]
        toplam = gt.get("total", {})
        toplam = toplam.get("total_price") if isinstance(toplam, dict) else None
        if isinstance(toplam, str) and re.search(r"\d", toplam):
            fisler.append((f"cord_{i}", ex["image"].convert("RGB"), toplam))
        if len(fisler) == FIS_SAYISI:
            break
else:
    from google.colab import files
    yuklenen = files.upload()
    for ad, veri in yuklenen.items():
        fisler.append((ad, Image.open(io.BytesIO(veri)).convert("RGB"), DOGRU_TOPLAM.get(ad, "?")))

print(len(fisler), "fiş hazır")
if len(fisler) == 0:
    raise ValueError("Hiç fiş verisi bulunamadı! Lütfen kaynak seçimini veya dosya yüklemesini kontrol edin.")


In [ ]:
import time, gc, torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

# A100/L4 GPU'larda bfloat16, T4 GPU'larda daha kararlı ve hızlı olan float16 seçilir
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

def calistir(model_id):
    processor = AutoProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="cuda").eval()
    cevaplar, sureler = [], []
    prompt = "<image>answer en " + SORU
    for isim, img, _ in fisler:
        # girdi sözlüğü GPU'ya taşınır; input_ids int64 kalmalıdır, sadece pixel_values DTYPE'a çevrilir
        girdi = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
        if "pixel_values" in girdi:
            girdi["pixel_values"] = girdi["pixel_values"].to(DTYPE)
            
        n = girdi["input_ids"].shape[-1]
        t0 = time.time()
        with torch.inference_mode():
            cikti = model.generate(**girdi, max_new_tokens=20, do_sample=False)
        sureler.append(time.time() - t0)
        cevaplar.append(processor.decode(cikti[0][n:], skip_special_tokens=True).strip())
        
    del model, processor; gc.collect(); torch.cuda.empty_cache()
    return cevaplar, sureler

sonuc = {}
for px, mid in MODELLER.items():
    print("Çalışıyor:", mid)
    sonuc[px] = calistir(mid)


In [ ]:
import pandas as pd

def rakam(s):
    return re.sub(r"\D", "", s or "")

def dogru_mu(cevap, dogru):
    a, b = rakam(cevap), rakam(dogru)
    if not a or not b:
        return False
    # "60.000" ile "60,000" aynı; sondaki kuruş hanesi (",00") farkını da aynı say
    return a == b or a == b + "00" or b == a + "00"

satirlar = []
for k, (isim, _, dogru) in enumerate(fisler):
    satir = {"fiş": isim, "doğru toplam": dogru}
    for px in MODELLER:
        cevap = sonuc[px][0][k]
        satir[f"{px}px cevap"] = cevap
        satir[f"{px}px doğru mu"] = dogru_mu(cevap, dogru)
    satirlar.append(satir)
tablo = pd.DataFrame(satirlar)
display(tablo)

n = len(fisler)
d = {px: int(tablo[f"{px}px doğru mu"].sum()) for px in MODELLER}
ort = {px: (sum(sonuc[px][1]) / n) if n > 0 else 0 for px in MODELLER}
print({px: f"{d[px]}/{n} doğru, fiş başına {ort[px]:.2f} sn" for px in MODELLER})


In [ ]:
import matplotlib.pyplot as plt

sutun = min(5, max(1, n))
satir_sayisi = (n + sutun - 1) // sutun
fig, axes = plt.subplots(satir_sayisi, sutun, figsize=(3.2 * sutun, 5.2 * max(1, satir_sayisi)))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
for ax in axes:
    ax.axis("off")
for k, (isim, img, dogru) in enumerate(fisler):
    ax = axes[k]
    ax.imshow(img)
    renk = lambda ok: "#1E8E5A" if ok else "#C0392B"
    isaret = lambda ok: "✓" if ok else "✗"
    a224, a448 = sonuc[224][0][k], sonuc[448][0][k]
    o224, o448 = dogru_mu(a224, dogru), dogru_mu(a448, dogru)
    ax.set_title(f"Doğru: {dogru}", fontsize=10, color="#1F2937", pad=30)
    ax.text(0.5, 1.10, f"224px: {a224[:18]} {isaret(o224)}", transform=ax.transAxes, ha="center", fontsize=9.5, color=renk(o224))
    ax.text(0.5, 1.03, f"448px: {a448[:18]} {isaret(o448)}", transform=ax.transAxes, ha="center", fontsize=9.5, color=renk(o448))
fig.suptitle(f'Aynı soru: "{SORU}"   ·   224px: {d[224]}/{n} doğru   ·   448px: {d[448]}/{n} doğru', fontsize=13, y=1.0)
fig.tight_layout()
fig.savefig("fis_karsilastirma.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

try:
    from google.colab import files
    files.download("fis_karsilastirma.png")
except Exception:
    pass


## Yazıya yapıştırılacak cümleler
Aşağıdaki iki cümleyi Word dosyasındaki sarı işaretli yerlere yapıştır.

In [ ]:
kaynak = f"CORD veri setinden {n} gerçek fiş" if KAYNAK == "cord" else f"Kendi çektiğim {n} fiş"
sn = lambda x: f"{x:.1f}".replace(".", ",")
print("ŞEKİL 7 AÇIKLAMASI:")
print(f'Şekil 7. {kaynak}, aynı soru ("{SORU}"), iki çözünürlük. '
      f'PaliGemma 2 3B mix, 224 pikselde {d[224]}/{n}, 448 pikselde {d[448]}/{n} fişin toplamını doğru okudu.')
print()
print("SONUÇ CÜMLESİ:")
fark = d[448] - d[224]
if fark > 0:
    yorum = "Makaledeki DocVQA farkının küçük bir benzeri benim denememde de çıktı: göz keskinleşince fişin toplamı okunabilir hâle geliyor."
elif fark == 0:
    yorum = "Benim denememde iki çözünürlük arasında fark çıkmadı; fişler yeterince büyük ve netse 224 piksel de yetebiliyor."
else:
    yorum = "Benim denememde beklenenin tersi çıktı; bu kadar az fişle kesin bir şey söylemek zor."
print(f"Sonuç: {n} fişin 224 pikselde {d[224]} tanesinde, 448 pikselde ise {d[448]} tanesinde toplam doğru okundu. "
      f"448 piksel fiş başına {sn(ort[448])} saniye, 224 piksel {sn(ort[224])} saniye sürdü. {yorum}")